# M6 — The MCP server: one standard plug for all the data tools

**Goal of this step:** put every data tool behind one **MCP server**, so the agent (M7), or any other AI app,
can use them the same way, with the safety rules built in.

**Analogy:** MCP (Model Context Protocol) is like **USB-C for AI tools**. Before USB-C, every device needed its own
cable. With MCP, a tool is written once, and any AI app that "speaks MCP" can plug in, see what tools exist,
read what each one does, and call it.

**How it works**
1. The AI app starts the server (`python mcp_server/server.py`) and they talk over stdin/stdout.
2. The app asks **"what tools do you have?"** and gets each tool's name, description and inputs.
3. The AI decides which tool to call; the app sends the call; the server runs it and sends the result back.

**Our server "adskg" has 6 tools**
| Tool | Uses | What it answers |
|---|---|---|
| `search_tables` | Qdrant (M2) | which tables match this question? |
| `get_join_path` | Neo4j (M5) | how do I join table A to table B? |
| `list_lookalikes` | Neo4j (M5) | which similar tables could I confuse this with? |
| `describe_table` | table cards (M2) | what does this table mean, and what are its columns? |
| `sample_rows` | DuckDB (M1) | what do real rows look like? |
| `run_sql` | DuckDB + safety (M4) | run one read-only SELECT |

**Safety lives inside the server**, so no app can skip it: read-only database, one SELECT only, ≤ 1,000 rows,
20 s limit, only real table names. **Every call is logged** to `logs/mcp_calls.jsonl`.

**Files in this step**
| File | What it does |
|---|---|
| `mcp_server/server.py` | the server and its 6 tools (built with FastMCP) |
| `mcp_server/safety.py` | the safety rules (from M4) |
| `mcp_server/check_mcp.py` | done-test: starts the server like a real app would, calls every tool, tries 6 attacks |

**Done when:** all tools can be called through MCP and write attempts are blocked ✅
(LangGraph calling them is part of M7.)

In [1]:
import sys, json
for folder in ["../mcp_server", "../catalog"]:
    sys.path.append(folder)

from fastmcp import Client
from server import mcp       # the server object; Client(mcp) talks to it directly inside this notebook

/Users/anshikapuneet/Documents/Claude_history/capstone_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. What an AI app sees: the tool list
This is all an AI gets to decide which tool to use: a **name**, a **description** (from the function's docstring)
and the **inputs**. That's why the docstrings in `server.py` are written for the AI to read.

In [2]:
async with Client(mcp) as client:
    tools = await client.list_tools()
for t in tools:
    inputs = ", ".join(f"{name}: {spec.get('type')}" for name, spec in t.input_schema["properties"].items())
    print(f"{t.name}({inputs})\n    {t.description}\n")

search_tables(question: string, k: integer)
    Find the k tables whose description best matches the question (search by meaning).

get_join_path(from_table: string, to_table: string)
    Shortest route of joins between two tables (at most 3 steps), with the join column for each step.

list_lookalikes(table: string)
    Tables that look similar to this one but answer a different question, and what differs.

describe_table(table: string)
    The table's card: what it means, one row = ?, freshness, traffic, columns, dates, joins.

sample_rows(table: string, n: integer)
    A few real rows from a table (at most 20).

run_sql(sql: string)
    Run ONE read-only SELECT query (DuckDB SQL). At most 1,000 rows; stopped after 20 seconds.



## 2. Call each tool
Let's walk through answering *"How many unique people saw campaign 83237 in the last 6 hours?"* the way the
agent will in M7: search → check look-alikes → describe → run SQL.

In [3]:
async def call(name, **args):
    async with Client(mcp) as client:
        result = await client.call_tool(name, args)
    return result.data

for hit in await call("search_tables", question="How many unique people saw campaign 83237 in the last 6 hours?", k=3):
    print(f"{hit['score']:.3f}  {hit['table']}")

0.698  live_reach_campaign_last6h
0.679  live_campaign_last6h
0.667  reach_campaign_total


In [4]:
for t in await call("list_lookalikes", table="live_reach_campaign_last6h"):
    print(f"{t['table']:28s} differs by {t['differs_by']}")

live_campaign_last6h         differs by ['what is counted']
reach_campaign_daily         differs by ['grain', 'freshness']


In [5]:
card = await call("describe_table", table="live_reach_campaign_last6h")
print(card["description"])
print("columns:", [c["name"] for c in card["columns"]])

Live reach for the last 6 hours only (13 May 18:00-23:59) - unique users who saw and clicked each campaign right now.
columns: ['campaign_id', 'unique_users', 'unique_clickers']


In [6]:
await call("run_sql", sql="SELECT unique_users FROM live_reach_campaign_last6h WHERE campaign_id = 83237")

{'columns': ['unique_users'],
 'rows': [['8']],
 'row_count': 1,
 'truncated': False,
 'error': ''}

And the two map tools: a join route with the column for each step, and real sample rows.

In [7]:
route = await call("get_join_path", from_table="dim_user", to_table="agg_campaign_daily")
for step in route["steps"]:
    print(f"{step['from']:16s} -> {step['to']:20s} on {step['on']}")

dim_user         -> fact_ad_events       on user_id
fact_ad_events   -> dim_advertiser       on advertiser_id
dim_advertiser   -> agg_campaign_daily   on advertiser_id


In [8]:
await call("sample_rows", table="dim_campaign", n=3)

{'columns': ['campaign_id', 'advertiser_id', 'num_ads'],
 'rows': [['395195', '13', '4'], ['155966', '54', '3'], ['77444', '69', '1']],
 'error': ''}

## 3. Try to break it
Each of these must be refused. Note the last one: someone hides SQL inside a *table name*.

In [9]:
attacks = [
    ("run_sql", {"sql": "DROP TABLE dim_ad"}),
    ("run_sql", {"sql": "SELECT 1; DROP TABLE dim_ad"}),
    ("run_sql", {"sql": "COPY dim_ad TO 'stolen.csv'"}),
    ("sample_rows", {"table": "dim_ad; DROP TABLE dim_ad"}),
]
async with Client(mcp) as client:
    for name, args in attacks:
        result = await client.call_tool(name, args, raise_on_error=False)
        reason = result.content[0].text if result.is_error else result.data["error"]
        print(f"BLOCKED {name}{args}\n   -> {reason[:100]}")

BLOCKED run_sql{'sql': 'DROP TABLE dim_ad'}
   -> only SELECT queries are allowed
BLOCKED run_sql{'sql': 'SELECT 1; DROP TABLE dim_ad'}
   -> exactly one SQL statement is allowed
BLOCKED run_sql{'sql': "COPY dim_ad TO 'stolen.csv'"}
   -> only SELECT queries are allowed


[10/30/26 10:08:14] Error calling tool 'sample_rows'

BLOCKED sample_rows{'table': 'dim_ad; DROP TABLE dim_ad'}
   -> unknown table 'dim_ad; DROP TABLE dim_ad'. Use search_tables to find table names.


## 4. Everything is logged
Every call, whether it worked or was blocked, goes to `logs/mcp_calls.jsonl`. This is useful for debugging the
agent, and it gives an audit trail.

In [10]:
for line in open("../logs/mcp_calls.jsonl").readlines()[-5:]:
    print(line.strip()[:160])

{"time": "2026-10-30T10:08:14", "tool": "sample_rows", "args": {"table": "dim_campaign", "n": 3}, "seconds": 0.003, "error": ""}
{"time": "2026-10-30T10:08:14", "tool": "run_sql", "args": {"sql": "DROP TABLE dim_ad"}, "seconds": 0.0, "error": "only SELECT queries are allowed"}
{"time": "2026-10-30T10:08:14", "tool": "run_sql", "args": {"sql": "SELECT 1; DROP TABLE dim_ad"}, "seconds": 0.0, "error": "exactly one SQL statement is allowe
{"time": "2026-10-30T10:08:14", "tool": "run_sql", "args": {"sql": "COPY dim_ad TO 'stolen.csv'"}, "seconds": 0.0, "error": "only SELECT queries are allowed"}
{"time": "2026-10-30T10:08:14", "tool": "sample_rows", "args": {"table": "dim_ad; DROP TABLE dim_ad", "n": 5}, "seconds": 0.0, "error": "ToolError: unknown tabl


## 5. The real test: a separate program, the way an AI app connects
The cells above talk to the server inside this notebook. The done-test starts `server.py` as a **separate
program** over stdio, exactly as an AI app would. Run it from the terminal:
```
.venv/bin/python mcp_server/check_mcp.py
```
Result: **Tools working: 6/6, Attacks blocked: 6/6.**

## 6. Plugging in other AI apps
Any MCP app can use this server. For example, with Claude Code:
```
claude mcp add adskg -- /full/path/to/.venv/bin/python /full/path/to/mcp_server/server.py
```
Other apps (Cursor, Claude Desktop …) take the same command in their MCP settings. The safety rules come along
automatically, because they live inside the server.

## 7. Why this matters for the project
- **One tool layer for everything.** The agent (E, M7) and the stretch version (E′, with ready-made generic MCP
  servers) use the same plug, so the comparison is fair.
- **Safety in one place**, not copied into every version.
- **For your company:** the same pattern lets any internal AI assistant use the ads warehouse safely.

**Next step (M7):** version E, the LangGraph agent that *chooses* which of these tools to call, retries when SQL
fails, and remembers the conversation.